# ML-07 — Baseline Action Score and Top-20 Review

This notebook uses the March warehouse-derived page-level frame at `data/warehouse_extract/warehouse_model_frame.parquet`. The scoring rule and evaluation design remain unchanged from the starter-data version.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/Week1_RunTheStarterNotebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal sanity checks

Before encoding the baseline, I am checking two transparent signals against the observed decline label. The outcome used here is `is_declining_label = trend_direction == "down"`; it is only for auditing/evaluation, not as a ranking feature. I will not use `trend_direction` or `trend_pct` in the baseline score.

Signals checked:

- `days_since_last_update`: a staleness/freshness signal tied directly to FlyRank refresh flags.
- `ctr` among visible, valid-position pages: a low-click-through opportunity signal where impressions are high enough and `avg_position` is real (`avg_position > 0`).

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repo root from either the notebook folder or the repo root."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/warehouse_extract/warehouse_model_frame.parquet").exists():
            return candidate
    raise FileNotFoundError("Could not find data/warehouse_extract/warehouse_model_frame.parquet")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data/warehouse_extract/warehouse_model_frame.parquet"
OUTPUT_PATH = REPO_ROOT / "work/outputs/baseline_action_score.csv"

df = pd.read_parquet(DATA_PATH)

# Evaluation-only label. Do not use trend_direction or trend_pct as score inputs.
df["is_declining_label"] = (df["trend_direction"].eq("down")).astype(int)

print(f"Loaded {len(df):,} rows x {df.shape[1]} source columns from {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Observed decline base rate: {df['is_declining_label'].mean():.1%}")

schema = pd.DataFrame(
    {
        "column": df.drop(columns=["is_declining_label"]).columns,
        "dtype": [str(dtype) for dtype in df.drop(columns=["is_declining_label"]).dtypes],
        "missing": df.drop(columns=["is_declining_label"]).isna().sum().to_numpy(),
    }
)
display(schema)


def make_bucket_table(
    data: pd.DataFrame,
    signal_col: str,
    bins: list[float],
    labels: list[str],
    mask: pd.Series | None = None,
) -> pd.DataFrame:
    """Return bucket-level counts and decline rates for one candidate signal."""
    if mask is None:
        mask = pd.Series(True, index=data.index)

    audit = data.loc[mask, [signal_col, "is_declining_label"]].dropna().copy()
    audit["bin_range"] = pd.cut(
        audit[signal_col],
        bins=bins,
        labels=labels,
        include_lowest=True,
    )

    table = (
        audit.groupby("bin_range", observed=False)
        .agg(
            n=("is_declining_label", "size"),
            declining_rate=("is_declining_label", "mean"),
            signal_min=(signal_col, "min"),
            signal_max=(signal_col, "max"),
        )
        .reset_index()
    )
    table["declining_rate"] = table["declining_rate"].round(3)
    return table


# Signal 1: staleness / days since last update.
staleness_table = make_bucket_table(
    df,
    signal_col="days_since_last_update",
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"],
)

print("Signal: days_since_last_update")
display(staleness_table)
print(
    "Verdict: MIXED - decline rate is higher for 31-180 day stale pages than fresh pages, "
    "but the tiny 181+ bucket falls back down, so staleness is useful but not sufficient alone."
)


# Signal 2: CTR on visible pages with real ranking data.
# avg_position = 0 means no data, so exclude it from this CTR opportunity check.
visible_valid_position = (
    df["impressions_90d"].ge(300)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
)

ctr_table = make_bucket_table(
    df,
    signal_col="ctr",
    bins=[-np.inf, 0.10, 0.25, 0.50, 1.00, np.inf],
    labels=["<=0.10%", "0.11-0.25%", "0.26-0.50%", "0.51-1.00%", ">1.00%"],
    mask=visible_valid_position,
)

print("\nSignal: ctr on visible pages with valid avg_position")
print(f"Audit slice: {visible_valid_position.sum():,} rows with impressions_90d >= 300 and 0 < avg_position <= 20")
display(ctr_table)
print(
    "Verdict: CONFIRMED - lower CTR buckets have clearly higher observed decline rates in the visible-page slice, "
    "matching the FlyRank low-CTR opportunity flag."
)


LEAKAGE_FEATURES = {"trend_direction", "trend_pct", "is_declining_label"}
section_1_candidate_signals = {"days_since_last_update", "ctr", "impressions_90d", "avg_position"}
assert section_1_candidate_signals.isdisjoint(LEAKAGE_FEATURES), "Section 1 candidate signals include leakage columns."


Loaded 120,513 rows x 33 source columns from data/warehouse_extract/warehouse_model_frame.parquet
Observed decline base rate: 43.3%


,column,dtype,missing
0,client_id,object,0
1,content_id,object,0
2,impressions_90d,float64,0
3,clicks_90d,float64,0
4,sessions_90d,float64,0
5,avg_position,float64,0
6,days_with_impressions,int64,0
7,content_type,object,0
8,main_intent,object,3228
9,word_count,float64,40897


Signal: days_since_last_update


,bin_range,n,declining_rate,signal_min,signal_max
0,0-30 days,120046,0.432,0.0,20.0
1,31-90 days,137,0.664,38.0,65.0
2,91-180 days,291,0.526,97.0,175.0
3,181+ days,39,0.667,183.0,287.0


Verdict: MIXED - decline rate is higher for 31-180 day stale pages than fresh pages, but the tiny 181+ bucket falls back down, so staleness is useful but not sufficient alone.

Signal: ctr on visible pages with valid avg_position
Audit slice: 43,544 rows with impressions_90d >= 300 and 0 < avg_position <= 20


,bin_range,n,declining_rate,signal_min,signal_max
0,<=0.10%,12912,0.544,0.000000,0.100000
1,0.11-0.25%,10293,0.464,0.100100,0.250000
2,0.26-0.50%,10662,0.356,0.250069,0.500000
3,0.51-1.00%,7009,0.318,0.500203,1.000000
4,>1.00%,2668,0.245,1.001065,7.317073


Verdict: CONFIRMED - lower CTR buckets have clearly higher observed decline rates in the visible-page slice, matching the FlyRank low-CTR opportunity flag.


## 2. Build the ranked queue (writes the CSV)

Plain-English rule: prioritize pages that are already visible in search, have real position data, and appear to underperform on CTR for where they rank. Staleness and impression volume raise the priority because stale visible pages with poor CTR are more likely to be useful action items.

One rule output:

- `score`: continuous priority score from 0 to 100.
- `reason_code`: `REASON_LOW_CTR_VISIBLE_STALE`.
- `action_label`: `FIX_TITLE_META_REFRESH`.

Leakage guard: the score does not use `trend_direction`, `trend_pct`, `is_declining_label`, or any last/previous 30-day trend-window columns.

In [2]:
import json


def percentile_rank(series: pd.Series) -> pd.Series:
    """Convert a numeric series to a stable 0-1 percentile rank."""
    return series.rank(method="average", pct=True).fillna(0.0)


def expected_ctr_by_position(avg_position: pd.Series) -> pd.Series:
    """Simple hand-written CTR expectation for broad GSC position bands."""
    return pd.Series(
        np.select(
            [
                avg_position.between(0.01, 3, inclusive="both"),
                avg_position.between(3.01, 10, inclusive="both"),
                avg_position.between(10.01, 20, inclusive="both"),
            ],
            [2.00, 1.00, 0.50],
            default=np.nan,
        ),
        index=avg_position.index,
    )


score_inputs = {"ctr", "avg_position", "impressions_90d", "days_since_last_update"}
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}
assert score_inputs.isdisjoint(forbidden_inputs), "Baseline score uses leakage or trend-window inputs."

ranked_df = df.copy()
ranked_df["expected_ctr"] = expected_ctr_by_position(ranked_df["avg_position"])
ranked_df["visible_valid_position"] = (
    ranked_df["impressions_90d"].ge(300)
    & ranked_df["avg_position"].gt(0)
    & ranked_df["avg_position"].le(20)
)

ranked_df["ctr_gap_score"] = (
    (ranked_df["expected_ctr"] - ranked_df["ctr"]) / ranked_df["expected_ctr"]
).clip(lower=0, upper=1).fillna(0.0)
ranked_df["volume_score"] = percentile_rank(np.log1p(ranked_df["impressions_90d"]))
ranked_df["freshness_score"] = (ranked_df["days_since_last_update"] / 180).clip(lower=0, upper=1)

ranked_df["score"] = (
    ranked_df["visible_valid_position"].astype(float)
    * 100
    * (
        0.55 * ranked_df["ctr_gap_score"]
        + 0.30 * ranked_df["volume_score"]
        + 0.15 * ranked_df["freshness_score"]
    )
).round(4)

ranked_df["reason_code"] = "REASON_LOW_CTR_VISIBLE_STALE"
ranked_df["action_label"] = "FIX_TITLE_META_REFRESH"
ranked_df = ranked_df.sort_values(
    ["score", "impressions_90d", "ctr_gap_score"],
    ascending=[False, False, False],
).reset_index(drop=True)
ranked_df.insert(0, "baseline_rank", np.arange(1, len(ranked_df) + 1))

output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action_label",
    "is_declining_label",
    "ctr",
    "expected_ctr",
    "ctr_gap_score",
    "impressions_90d",
    "avg_position",
    "days_since_last_update",
    "freshness_score",
    "volume_score",
    "content_age_days",
    "content_type",
    "main_intent",
    "position_tier",
    "freshness_tier",
    "impression_tier",
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ranked_df[output_columns].to_csv(OUTPUT_PATH, index=False)

base_rate = ranked_df["is_declining_label"].mean()
precision_at_10 = ranked_df.head(10)["is_declining_label"].mean()
precision_at_50 = ranked_df.head(50)["is_declining_label"].mean()
precision_at_500 = ranked_df.head(500)["is_declining_label"].mean()

metrics = {
    "rows": int(len(ranked_df)),
    "output_path": str(OUTPUT_PATH.relative_to(REPO_ROOT)),
    "base_declining_rate": round(float(base_rate), 4),
    "precision_at_10": round(float(precision_at_10), 4),
    "precision_at_50": round(float(precision_at_50), 4),
    "precision_at_500": round(float(precision_at_500), 4),
    "score_inputs": sorted(score_inputs),
    "forbidden_inputs_used": sorted(score_inputs.intersection(forbidden_inputs)),
    "reason_code": "REASON_LOW_CTR_VISIBLE_STALE",
    "action_label": "FIX_TITLE_META_REFRESH",
}

metrics_path = OUTPUT_PATH.with_name("baseline_action_score_metrics.json")
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

print(f"Wrote ranked queue: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote metrics receipt: {metrics_path.relative_to(REPO_ROOT)}")
print(f"Base decline rate: {base_rate:.1%}")
print(f"Top-10 precision: {precision_at_10:.1%}")
print(f"Top-50 precision: {precision_at_50:.1%}")
print(f"Top-500 precision: {precision_at_500:.1%}")
display(ranked_df[output_columns].head(10).drop(columns=["content_id", "client_id"]))


Wrote ranked queue: work/outputs/baseline_action_score.csv
Wrote metrics receipt: work/outputs/baseline_action_score_metrics.json
Base decline rate: 43.3%
Top-10 precision: 80.0%
Top-50 precision: 76.0%


,baseline_rank,score,reason_code,action_label,is_declining_label,ctr,expected_ctr,ctr_gap_score,impressions_90d,avg_position,days_since_last_update,freshness_score,volume_score,content_age_days,content_type,main_intent,position_tier,freshness_tier,impression_tier
0,1,96.2507,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,1.0,1.000000,2043.0,5.687620,216.0,1.0,0.875022,216,keyword article,None,top_10,high_activity,high
1,2,94.8315,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,1.0,1.000000,1413.0,6.998111,231.0,1.0,0.827716,231,feedly article,None,top_10,medium_activity,high
2,3,90.4117,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,0.5,1.000000,2133.0,11.414193,108.0,0.6,0.880390,145,keyword article,informational,top_20,high_activity,high
3,4,86.4978,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,1.0,1.000000,83772.0,8.607910,18.0,0.1,0.999925,227,keyword article,informational,top_10,high_activity,high
4,5,86.3325,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,0,0.000000,1.0,1.000000,17150.0,8.160914,18.0,0.1,0.994416,262,keyword article,informational,top_10,high_activity,high
5,6,86.1502,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,1.0,1.000000,11830.0,7.278427,18.0,0.1,0.988342,262,keyword article,informational,top_10,high_activity,high
6,7,85.9001,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,2.0,1.000000,8569.0,1.430659,18.0,0.1,0.980002,170,keyword article,transactional,top_3,high_activity,high
7,8,85.8485,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.007179,1.0,0.992821,13929.0,4.655644,18.0,0.1,0.991445,227,keyword article,transactional,top_10,high_activity,high
8,9,85.8192,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,1,0.000000,0.5,1.000000,770.0,19.255662,108.0,0.6,0.727307,145,keyword article,informational,top_20,high_activity,medium
9,10,85.6072,REASON_LOW_CTR_VISIBLE_STALE,FIX_TITLE_META_REFRESH,0,0.000000,1.0,1.000000,6676.0,5.998120,18.0,0.1,0.970240,58,keyword article,informational,top_10,high_activity,high


## 3. Top-10 review

For each of the top ten, I am reading the rule with a skeptic's eye: what action it recommends, why the score is high, and what real-world counter-example could make the recommendation wrong.

In [3]:
def format_top10_reason(row: pd.Series) -> str:
    return (
        f"visible page with {row['impressions_90d']:,.0f} impressions, "
        f"position {row['avg_position']:.1f}, CTR {row['ctr']:.2f}% vs "
        f"{row['expected_ctr']:.2f}% expected, last updated {row['days_since_last_update']:.0f} days ago"
    )


def what_would_make_wrong(row: pd.Series) -> str:
    if row["avg_position"] <= 3 and row["ctr"] <= 0.10:
        return "SERP layout, brand mismatch, or query intent could suppress CTR even with a good title."
    if row["days_since_last_update"] <= 30:
        return "A recent update may not have had enough time to be reflected in search behavior."
    if row["main_intent"] == "transactional":
        return "Transactional intent may need offer/pricing work, not only title/meta refresh."
    if row["impressions_90d"] < 1000:
        return "The opportunity may be too small for the team to prioritize this week."
    return "The page may satisfy a narrow intent where low CTR is acceptable or caused by SERP features."


top10_review = ranked_df.head(10).copy()
top10_lines = []
for _, row in top10_review.iterrows():
    top10_lines.append(
        {
            "rank": int(row["baseline_rank"]),
            "action": row["action_label"],
            "reason_for_high_score": format_top10_reason(row),
            "what_would_make_it_wrong": what_would_make_wrong(row),
        }
    )

top10_review_table = pd.DataFrame(top10_lines)
display(top10_review_table)

for item in top10_lines:
    print(
        f"#{item['rank']}: {item['action']} | {item['reason_for_high_score']} | "
        f"Wrong if: {item['what_would_make_it_wrong']}"
    )


,rank,action,reason_for_high_score,what_would_make_it_wrong
0,1,FIX_TITLE_META_REFRESH,"visible page with 2,043 impressions, position ...",The page may satisfy a narrow intent where low...
1,2,FIX_TITLE_META_REFRESH,"visible page with 1,413 impressions, position ...",The page may satisfy a narrow intent where low...
2,3,FIX_TITLE_META_REFRESH,"visible page with 2,133 impressions, position ...",The page may satisfy a narrow intent where low...
3,4,FIX_TITLE_META_REFRESH,"visible page with 83,772 impressions, position...",A recent update may not have had enough time t...
4,5,FIX_TITLE_META_REFRESH,"visible page with 17,150 impressions, position...",A recent update may not have had enough time t...
5,6,FIX_TITLE_META_REFRESH,"visible page with 11,830 impressions, position...",A recent update may not have had enough time t...
6,7,FIX_TITLE_META_REFRESH,"visible page with 8,569 impressions, position ...","SERP layout, brand mismatch, or query intent c..."
7,8,FIX_TITLE_META_REFRESH,"visible page with 13,929 impressions, position...",A recent update may not have had enough time t...
8,9,FIX_TITLE_META_REFRESH,"visible page with 770 impressions, position 19...",The opportunity may be too small for the team ...
9,10,FIX_TITLE_META_REFRESH,"visible page with 6,676 impressions, position ...",A recent update may not have had enough time t...


#1: FIX_TITLE_META_REFRESH | visible page with 2,043 impressions, position 5.7, CTR 0.00% vs 1.00% expected, last updated 216 days ago | Wrong if: The page may satisfy a narrow intent where low CTR is acceptable or caused by SERP features.
#2: FIX_TITLE_META_REFRESH | visible page with 1,413 impressions, position 7.0, CTR 0.00% vs 1.00% expected, last updated 231 days ago | Wrong if: The page may satisfy a narrow intent where low CTR is acceptable or caused by SERP features.
#3: FIX_TITLE_META_REFRESH | visible page with 2,133 impressions, position 11.4, CTR 0.00% vs 0.50% expected, last updated 108 days ago | Wrong if: The page may satisfy a narrow intent where low CTR is acceptable or caused by SERP features.
#4: FIX_TITLE_META_REFRESH | visible page with 83,772 impressions, position 8.6, CTR 0.00% vs 1.00% expected, last updated 18 days ago | Wrong if: A recent update may not have had enough time to be reflected in search behavior.
#5: FIX_TITLE_META_REFRESH | visible page with 17,1

## 4. Weak picks + leakage check

The baseline should have weak picks; if it did not, I would not trust the review. I am checking mid-tier/tail candidates and then asserting the output path, schema, sorting, and leakage constraints.

In [4]:
positive_scores = ranked_df.loc[ranked_df["score"].gt(0)].copy()
mid_low_candidates = positive_scores.loc[
    (positive_scores["baseline_rank"] >= positive_scores["baseline_rank"].quantile(0.45))
    & (positive_scores["baseline_rank"] <= positive_scores["baseline_rank"].quantile(0.65))
].copy()

weak_pick_table = (
    mid_low_candidates.assign(
        low_confidence_reason=np.select(
            [
                mid_low_candidates["ctr_gap_score"].lt(0.20),
                mid_low_candidates["days_since_last_update"].le(30),
                mid_low_candidates["impressions_90d"].lt(1000),
            ],
            [
                "CTR is only slightly below the hand-written expectation.",
                "Page was updated recently, so intervention may be premature.",
                "Search demand is modest even if the rule sees an opportunity.",
            ],
            default="Mid-tier score: plausible, but not as urgent as the top queue.",
        )
    )
    .loc[
        :,
        [
            "baseline_rank",
            "score",
            "action_label",
            "reason_code",
            "impressions_90d",
            "avg_position",
            "ctr",
            "days_since_last_update",
            "low_confidence_reason",
        ],
    ]
    .head(10)
)

display(weak_pick_table)

required_columns = {
    "baseline_rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action_label",
}
written = pd.read_csv(OUTPUT_PATH)

assert OUTPUT_PATH.exists(), f"Missing output CSV: {OUTPUT_PATH}"
assert metrics_path.exists(), f"Missing metrics JSON: {metrics_path}"
assert required_columns.issubset(written.columns), "Output CSV is missing required columns."
assert len(written) == len(df), "Output CSV should rank every source row."
assert written["score"].notna().all(), "Score contains null values."
assert np.isfinite(written["score"]).all(), "Score contains non-finite values."
assert written["score"].is_monotonic_decreasing, "Output CSV is not sorted by descending score."
assert written["reason_code"].nunique() == 1, "This assignment asks for one baseline reason code."
assert written["action_label"].nunique() == 1, "This assignment asks for one baseline action label."
assert score_inputs.isdisjoint(forbidden_inputs), "Forbidden leakage inputs were used in the score."

print("Verification passed:")
print(f"- CSV path exists: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"- Metrics JSON exists: {metrics_path.relative_to(REPO_ROOT)}")
print(f"- Rows ranked: {len(written):,}")
print(f"- Required schema present: {sorted(required_columns)}")
print(f"- Score inputs: {sorted(score_inputs)}")
print(f"- Leakage inputs used: {sorted(score_inputs.intersection(forbidden_inputs))}")


,baseline_rank,score,action_label,reason_code,impressions_90d,avg_position,ctr,days_since_last_update,low_confidence_reason
19595,19596,69.4413,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,2382.0,14.882577,0.125945,18.0,"Page was updated recently, so intervention may..."
19596,19597,69.4404,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,755.0,2.928301,0.264901,0.0,"Page was updated recently, so intervention may..."
19597,19598,69.4404,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,755.0,6.442117,0.132450,0.0,"Page was updated recently, so intervention may..."
19598,19599,69.4404,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,755.0,4.992346,0.132450,0.0,"Page was updated recently, so intervention may..."
19599,19600,69.4392,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,956.0,1.756079,0.313808,0.0,"Page was updated recently, so intervention may..."
19600,19601,69.4382,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,1157.0,3.885214,0.172861,0.0,"Page was updated recently, so intervention may..."
19601,19602,69.4374,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,3650.0,9.810010,0.246575,0.0,"Page was updated recently, so intervention may..."
19602,19603,69.4364,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,1902.0,6.263230,0.210305,0.0,"Page was updated recently, so intervention may..."
19603,19604,69.4350,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,2259.0,9.958342,0.221337,0.0,"Page was updated recently, so intervention may..."
19604,19605,69.4344,FIX_TITLE_META_REFRESH,REASON_LOW_CTR_VISIBLE_STALE,516.0,2.472034,0.193798,0.0,"Page was updated recently, so intervention may..."


Verification passed:
- CSV path exists: work/outputs/baseline_action_score.csv
- Metrics JSON exists: work/outputs/baseline_action_score_metrics.json
- Rows ranked: 120,513
- Required schema present: ['action_label', 'baseline_rank', 'client_id', 'content_id', 'reason_code', 'score']
- Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
- Leakage inputs used: []


## 5. Self-check

Before submitting, I can confirm:

- [x] Every section above is filled with markdown thinking and code that backs it.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] No client names, URLs, or private queries are printed.
- [x] The baseline uses careful decision-support language and reports observed rates.
- [x] The score avoids target leakage from `trend_direction`, `trend_pct`, and trend-window columns.
- [x] Final human step: run the notebook top to bottom one more time, commit `work/notebooks/w04_baseline_score.ipynb`, and submit the repo URL.